# 06 — Z-scored BV category embeddings from exemplar crops (valid129 / valid85)

> **Stage 0a** — Run before notebooks **02–05**. See [`00_build_exemplar_embeddings.md`](00_build_exemplar_embeddings.md) (step 0a), then notebook **07** (THINGS). Check outputs: `python scripts/check_exemplar_stage.py`.

Recompute **BabyView** category-level embeddings by **averaging precomputed per-image `.npy` embeddings**, then **z-scoring each dimension across categories** (mean 0, std 1 per column), **within** each category set only (not the legacy 163-cat hierarchical export).

**Exemplar selection**
- **valid129:** Only embeddings listed in **`BV_CLIP_FILTER_LIST`** (default: `clip_image_embeddings_filtered-by-clip-0.27_exclude-people_exclude-subject-00270001.txt` under `BV_EMBEDDINGS_BASE`; filename token follows `BV_CLIP_FILTER_LIST_THRESHOLD`). Each line is a full path to a CLIP `.npy`; we keep `(category, basename)` pairs whose category is in the inclusion list, then require the **same basename** under both flat model dirs (`clip_embeddings_new` / `facebook_dinov3-vitb16-pretrain-lvd1689m`). **Not** YOLOE score in the filename (that is detection confidence, not CLIP).
- **valid85:** Same gates as notebook **03** (sampled manifest ∩ per-file; categories = inclusion ∩ per-class). Resolve `.npy` from manifest stems **only if** `(category, stem)` appears in **`BV_CLIP_FILTER_LIST`** (same CLIP-pass list as valid129).

**Precomputed embeddings:** load BabyDINOv3 `.npy` vectors from `babydinov3_grad_accum_1/step_{checkpoint}` (default step `119999`), restricted by the same **CLIP filter list** as notebook 06 originally used for exemplar selection. L2-normalize each vector, average per category, z-score across categories.

**CLIP / facebook DINOv3:** commented out in the main loop (already computed in prior runs). Re-enable those blocks if you need to refresh `bv_clip_*` / `bv_dinov3_*` CSVs.

**Run:** from `analysis/manuscript-2026/` (`cd analysis/manuscript-2026` so `Path.cwd().parents[1]` is the repo root). Requires `pandas`, `numpy`, `tqdm` (no GPU).

**Tmux (disconnect-safe):** `./run_06_zscore_tmux.sh` runs `exemplar_set_zscore_embeddings.py` in session `bv_zscore06` with a log under `logs/`. Attach with `tmux attach -t bv_zscore06` or `tail -f logs/exemplar_set_zscore_*.log`.

**Runtime (valid129):** scanning the CLIP filter list (~4.2M lines) plus loading ~3.6M `.npy` files per backbone takes **tens of minutes to ~1+ hour** on shared storage. Progress bars show scan and per-category averaging. Run interactively in the notebook or in tmux; set `BV_INCLUDE_BABYDINOV3=0` to skip BabyDINOv3 and save one full pass.

Run conventions (aligned with other preprint notebooks):
- Set `CATEGORY_SET` in the setup cell to `"valid129"` (default) or `"valid85"`.
- Outputs go to `analysis/manuscript-2026/exemplar_set_embeddings/valid129/` or `analysis/manuscript-2026/exemplar_set_embeddings/valid85/`.

**Env:**
- `BV_EMBEDDINGS_BASE` — optional parent (default `/data2/.../yoloe_cdi_embeddings`); used only if the two dirs below are not set explicitly.
- `BV_CLIP_EMBEDDINGS_DIR` / `BV_DINOV3_EMBEDDINGS_DIR` — **flat** per-image roots (defaults under `BV_EMBEDDINGS_BASE`). Do **not** point at `*_grouped_by_age-mo_*` outputs.
- `BV_INCLUDE_BABYDINOV3` — `1`/`0` (default `1`) to also process BabyDINOv3 checkpoint embeddings.
- `BV_BABYDINOV3_EMBEDDINGS_DIR` — flat per-image root (default `{BV_EMBEDDINGS_BASE}/babydinov3_grad_accum_1/step_{BV_BABYDINOV3_CHECKPOINT_STEP}`).
- `BV_BABYDINOV3_CHECKPOINT_STEP` — checkpoint folder name token (default `119999`).
- `BV_CLIP_FILTER_LIST` — full path to `clip_image_embeddings_filtered-by-clip-*_exclude-people_exclude-subject-00270001.txt` (optional; default derived from `BV_CLIP_FILTER_LIST_THRESHOLD` + `BV_EMBEDDINGS_BASE`).
- `BV_CLIP_FILTER_LIST_THRESHOLD` — token in that default filename only (default `0.27`; ignored if `BV_CLIP_FILTER_LIST` is set).
- `BV_CROP_PATH_PREFIX` / `BV_CROP_PATH_PREFIX_NEW` — optional prefix rewrite on **absolute** paths (manifest JPGs and resolved `.npy` paths).

**Outputs:** `exemplar_set_embeddings/{valid129,valid85}/` — `bv_babydinov3_exemplar_avg_zscore_within_{CATEGORY_SET}.csv`, raw companion, plus `exemplar_embedding_run.json`.

**THINGS (same category sets):** run `07_things_exemplar_set_zscore_embeddings.ipynb` — writes `things_{clip|dinov3}_exemplar_avg_*` alongside the `bv_*` files in the same folders.


In [1]:
import json
import os
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Resolve project root from this notebook location: analysis/manuscript-2026/
PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_DIR = PROJECT_ROOT / "data"
ANNOTATION_DIR = PROJECT_ROOT / "annotation"
PREPRINT_DIR = PROJECT_ROOT / "analysis" / "manuscript-2026"
OUT_ROOT = PREPRINT_DIR / "exemplar_set_embeddings"

CATEGORY_FILES = {
    "valid85": DATA_DIR / "included_categories_valid85.txt",
    "valid129": DATA_DIR / "included_categories_valid129.txt",
}

# Set this once at the top of the notebook (or export BV_CATEGORY_SET=valid85).
# Supported values: "valid129" (default) or "valid85"
CATEGORY_SET = os.environ.get("BV_CATEGORY_SET", "valid129")
if CATEGORY_SET not in CATEGORY_FILES:
    raise ValueError(
        f"Unsupported CATEGORY_SET: {CATEGORY_SET!r} (expected one of {sorted(CATEGORY_FILES)})"
    )
INCLUDED_CATEGORIES_TXT = CATEGORY_FILES[CATEGORY_SET]
OUTPUT_DIR = OUT_ROOT / CATEGORY_SET
# Backward-compatible alias for downstream cells that still iterate a list.
CATEGORY_SETS = [CATEGORY_SET]

PER_CLASS_PRECISION_CSV = ANNOTATION_DIR / "per_class_validation_data.csv"
PER_FILE_PRECISION_CSV = ANNOTATION_DIR / "per_file_precision_data.csv"
SAMPLED_EXEMPLAR_CSV = ANNOTATION_DIR / (
    "sampled_object_crops_100_bucket_assignments_100ex_8subj_per_video_cap_babyview_only.csv"
)
PRECISION_THRESHOLD = float(os.environ.get("BV_PRECISION_THRESHOLD", "0.6"))
CROP_PREFIX = os.environ.get("BV_CROP_PATH_PREFIX", "").strip()
CROP_PREFIX_NEW = os.environ.get("BV_CROP_PATH_PREFIX_NEW", "").strip()

_EMB_BASE = Path(
    os.environ.get(
        "BV_EMBEDDINGS_BASE",
        "/data2/dataset/babyview/868_hours/outputs/yoloe_cdi_embeddings",
    )
).expanduser()
CLIP_EMBEDDINGS_DIR = Path(
    os.environ.get("BV_CLIP_EMBEDDINGS_DIR", str(_EMB_BASE / "clip_embeddings_new"))
).expanduser()
DINOV3_EMBEDDINGS_DIR = Path(
    os.environ.get(
        "BV_DINOV3_EMBEDDINGS_DIR",
        str(_EMB_BASE / "facebook_dinov3-vitb16-pretrain-lvd1689m"),
    )
).expanduser()

# Main loop runs BabyDINOv3 only (CLIP/DINOv3 blocks commented out below).
RUN_BABYDINOV3_ONLY = True
INCLUDE_BABYDINOV3 = True
BABYDINOV3_CHECKPOINT_STEP = os.environ.get("BV_BABYDINOV3_CHECKPOINT_STEP", "119999").strip()
_default_babydinov3_dir = _EMB_BASE / "babydinov3_grad_accum_1" / f"step_{BABYDINOV3_CHECKPOINT_STEP}"
BABYDINOV3_EMBEDDINGS_DIR = Path(
    os.environ.get("BV_BABYDINOV3_EMBEDDINGS_DIR", str(_default_babydinov3_dir))
).expanduser()

CLIP_FILTER_LIST_THRESHOLD = os.environ.get("BV_CLIP_FILTER_LIST_THRESHOLD", "0.27").strip()
CLIP_FILTER_LIST_PATH = Path(
    os.environ.get(
        "BV_CLIP_FILTER_LIST",
        str(
            _EMB_BASE
            / f"clip_image_embeddings_filtered-by-clip-{CLIP_FILTER_LIST_THRESHOLD}_exclude-people_exclude-subject-00270001.txt"
        ),
    )
).expanduser()

print(f"[06_exemplar_set_zscore_embeddings] CATEGORY_SET={CATEGORY_SET!r}")
print(f"Included categories txt: {INCLUDED_CATEGORIES_TXT}")
print("PRECISION_THRESHOLD", PRECISION_THRESHOLD)
print("CLIP_FILTER_LIST_PATH", CLIP_FILTER_LIST_PATH, "exists:", CLIP_FILTER_LIST_PATH.is_file())
print("CLIP_FILTER_LIST_THRESHOLD", CLIP_FILTER_LIST_THRESHOLD)
print("RUN_BABYDINOV3_ONLY", RUN_BABYDINOV3_ONLY)
# print("CLIP_EMBEDDINGS_DIR", CLIP_EMBEDDINGS_DIR, "exists:", CLIP_EMBEDDINGS_DIR.is_dir())
# print("DINOV3_EMBEDDINGS_DIR", DINOV3_EMBEDDINGS_DIR, "exists:", DINOV3_EMBEDDINGS_DIR.is_dir())
if INCLUDE_BABYDINOV3:
    print("BABYDINOV3_CHECKPOINT_STEP", BABYDINOV3_CHECKPOINT_STEP)
    print(
        "BABYDINOV3_EMBEDDINGS_DIR",
        BABYDINOV3_EMBEDDINGS_DIR,
        "exists:",
        BABYDINOV3_EMBEDDINGS_DIR.is_dir(),
    )
print(f"Output dir: {OUTPUT_DIR}")


[06_exemplar_set_zscore_embeddings] CATEGORY_SET='valid129'
Included categories txt: /home/j7yang/babyview-projects/vss2026/object-detection/data/included_categories_valid129.txt
PRECISION_THRESHOLD 0.6
CLIP_FILTER_LIST_PATH /data2/dataset/babyview/868_hours/outputs/yoloe_cdi_embeddings/clip_image_embeddings_filtered-by-clip-0.27_exclude-people_exclude-subject-00270001.txt exists: True
CLIP_FILTER_LIST_THRESHOLD 0.27
RUN_BABYDINOV3_ONLY True
BABYDINOV3_CHECKPOINT_STEP 119999
BABYDINOV3_EMBEDDINGS_DIR /data2/dataset/babyview/868_hours/outputs/yoloe_cdi_embeddings/babydinov3_grad_accum_1/step_119999 exists: True
Output dir: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129


/data/j7yang/anaconda3/envs/vislearnlabpy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from collections import defaultdict

def resolve_category_subdir(embed_root: Path, cat: str) -> Path | None:
    direct = embed_root / cat
    if direct.is_dir():
        return direct
    for p in embed_root.iterdir():
        if p.is_dir() and p.name.lower() == cat.lower():
            return p
    return None


def build_category_subdir_cache(
    embed_root: Path,
    allowed_categories: set[str],
    label: str = "embed",
) -> dict[str, Path]:
    """Resolve category subdirs once (avoids millions of `iterdir` calls during filter-list scan)."""
    cache: dict[str, Path] = {}
    for cat in tqdm(sorted(allowed_categories), desc=f"Index {label} category dirs"):
        sub = resolve_category_subdir(embed_root, cat)
        if sub is not None:
            cache[cat] = sub
    return cache


def load_included_categories(txt_path: Path) -> list[str]:
    return [line.strip().lower() for line in txt_path.read_text().splitlines() if line.strip()]


def load_valid_classes(per_class_csv: Path, threshold: float) -> set[str]:
    df = pd.read_csv(per_class_csv, usecols=["class", "precision"])
    df["class"] = df["class"].astype(str).str.strip().str.lower()
    return set(df.loc[df["precision"] > threshold, "class"])


def load_valid_pairs(per_file_csv: Path, threshold: float) -> set[tuple[str, str]]:
    df = pd.read_csv(per_file_csv, usecols=["filename", "class", "precision"])
    df = df[df["precision"] > threshold].copy()
    df["class_norm"] = df["class"].astype(str).str.strip().str.lower()
    df["stem"] = (
        df["filename"]
        .astype(str)
        .str.strip()
        .str.rsplit("/", n=1)
        .str[-1]
        .str.rsplit(".", n=1)
        .str[0]
        .str.lower()
    )
    return set(zip(df["class_norm"], df["stem"]))


def remap_absolute_path(p: Path) -> Path:
    s = str(p)
    if CROP_PREFIX and CROP_PREFIX_NEW and s.startswith(CROP_PREFIX):
        return Path(CROP_PREFIX_NEW + s[len(CROP_PREFIX) :])
    return p


def load_clip_filter_pair_set(filter_list_path: Path, categories_lower: set[str]) -> set[tuple[str, str]]:
    """Same convention as preprocessing/sample_object_crops_variability.load_filtered_stems (CLIP-pass list)."""
    allowed: set[tuple[str, str]] = set()
    if not filter_list_path.is_file():
        raise FileNotFoundError(f"CLIP filter list not found: {filter_list_path}")
    with filter_list_path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            p = Path(line)
            if p.suffix.lower() == ".npy":
                stem = p.stem
            else:
                stem = p.name
            category = p.parent.name.strip().lower()
            if category in categories_lower:
                allowed.add((category, stem.lower()))
    return allowed


def collect_paired_npy_paths_from_clip_filter_list(
    filter_list_path: Path,
    clip_embed_root: Path,
    dino_embed_root: Path,
    allowed_categories: set[str],
) -> tuple[dict[str, list[Path]], dict[str, list[Path]], dict]:
    """Keep only stems listed in CLIP filter file; require same `.npy` basename under both flat model dirs."""
    clip_out: dict[str, list[Path]] = {}
    dino_out: dict[str, list[Path]] = {}
    stats = {
        "n_lines_in_clip_filter_list": 0,
        "n_lines_in_allowed_categories": 0,
        "n_pairs_both_models_on_disk": 0,
        "n_lines_missing_clip_or_dino_file": 0,
        "categories_with_subdir_clip": 0,
        "categories_with_subdir_dino": 0,
    }
    seen: dict[str, set[str]] = defaultdict(set)
    clip_subdirs = build_category_subdir_cache(clip_embed_root, allowed_categories, "CLIP")
    dino_subdirs = build_category_subdir_cache(dino_embed_root, allowed_categories, "DINOv3")

    with filter_list_path.open("r") as f:
        for line in tqdm(f, desc="Scan CLIP filter list (CLIP+DINO pair)", unit=" lines", mininterval=2.0):
            line = line.strip()
            if not line:
                continue
            stats["n_lines_in_clip_filter_list"] += 1
            p = Path(line)
            if p.suffix.lower() != ".npy":
                continue
            fname = p.name
            cat = p.parent.name.strip().lower()
            if cat not in allowed_categories:
                continue
            stats["n_lines_in_allowed_categories"] += 1
            key = fname.lower()
            if key in seen[cat]:
                continue
            seen[cat].add(key)

            cdir_c = clip_subdirs.get(cat)
            cdir_d = dino_subdirs.get(cat)
            pc = cdir_c / fname if cdir_c is not None else None
            pd = cdir_d / fname if cdir_d is not None else None
            if cdir_c is None or cdir_d is None or pc is None or pd is None:
                stats["n_lines_missing_clip_or_dino_file"] += 1
                continue
            if not pc.is_file() or not pd.is_file():
                stats["n_lines_missing_clip_or_dino_file"] += 1
                continue
            if cat not in clip_out:
                clip_out[cat] = []
                dino_out[cat] = []
            clip_out[cat].append(remap_absolute_path(pc))
            dino_out[cat].append(remap_absolute_path(pd))
            stats["n_pairs_both_models_on_disk"] += 1

    stats["categories_with_subdir_clip"] = len(clip_subdirs)
    stats["categories_with_subdir_dino"] = len(dino_subdirs)

    for cat in sorted(allowed_categories):
        clip_out.setdefault(cat, [])
        dino_out.setdefault(cat, [])
        clip_out[cat] = sorted(dict.fromkeys(clip_out[cat]), key=lambda x: str(x))
        dino_out[cat] = sorted(dict.fromkeys(dino_out[cat]), key=lambda x: str(x))

    print(
        f"  [CLIP filter list] lines={stats['n_lines_in_clip_filter_list']:,} "
        f"in_allowed_cats={stats['n_lines_in_allowed_categories']:,} "
        f"paired_on_disk={stats['n_pairs_both_models_on_disk']:,} "
        f"missing_file_or_dir={stats['n_lines_missing_clip_or_dino_file']:,}"
    )
    return clip_out, dino_out, stats


def collect_npy_paths_from_clip_filter_list(
    filter_list_path: Path,
    embed_root: Path,
    allowed_categories: set[str],
    model_label: str = "embed",
) -> tuple[dict[str, list[Path]], dict]:
    """Keep CLIP-filter-list stems; require `.npy` basename under a single flat embed root."""
    out: dict[str, list[Path]] = {}
    stats = {
        "model_label": model_label,
        "n_lines_in_clip_filter_list": 0,
        "n_lines_in_allowed_categories": 0,
        "n_paths_on_disk": 0,
        "n_lines_missing_file_or_dir": 0,
        "categories_with_subdir": 0,
    }
    seen: dict[str, set[str]] = defaultdict(set)
    subdirs = build_category_subdir_cache(embed_root, allowed_categories, model_label)

    with filter_list_path.open("r") as f:
        for line in tqdm(
            f,
            desc=f"Scan CLIP filter list ({model_label})",
            unit=" lines",
            mininterval=2.0,
        ):
            line = line.strip()
            if not line:
                continue
            stats["n_lines_in_clip_filter_list"] += 1
            p = Path(line)
            if p.suffix.lower() != ".npy":
                continue
            fname = p.name
            cat = p.parent.name.strip().lower()
            if cat not in allowed_categories:
                continue
            stats["n_lines_in_allowed_categories"] += 1
            key = fname.lower()
            if key in seen[cat]:
                continue
            seen[cat].add(key)

            cdir = subdirs.get(cat)
            pn = cdir / fname if cdir is not None else None
            if cdir is None or pn is None or not pn.is_file():
                stats["n_lines_missing_file_or_dir"] += 1
                continue
            out.setdefault(cat, []).append(remap_absolute_path(pn))
            stats["n_paths_on_disk"] += 1

    stats["categories_with_subdir"] = len(subdirs)
    for cat in sorted(allowed_categories):
        out.setdefault(cat, [])
        out[cat] = sorted(dict.fromkeys(out[cat]), key=lambda x: str(x))

    print(
        f"  [CLIP filter list → {model_label}] lines={stats['n_lines_in_clip_filter_list']:,} "
        f"in_allowed_cats={stats['n_lines_in_allowed_categories']:,} "
        f"on_disk={stats['n_paths_on_disk']:,} "
        f"missing_file_or_dir={stats['n_lines_missing_file_or_dir']:,}"
    )
    return out, stats


def valid85_npy_paths_by_category_from_manifest(
    df: pd.DataFrame,
    embed_root: Path,
    clip_filter_pairs: set[tuple[str, str]],
) -> dict[str, list[Path]]:
    d: dict[str, list[Path]] = defaultdict(list)
    for _, row in df.iterrows():
        cat = str(row["category"]).strip().lower()
        stem = str(row["stem"]).strip().lower()
        if (cat, stem) not in clip_filter_pairs:
            continue
        npy = remap_absolute_path(embed_root / cat / f"{stem}.npy")
        d[cat].append(npy)
    return {k: list(dict.fromkeys(v)) for k, v in sorted(d.items())}


def build_valid85_sampled_exemplar_table(
    included_txt: Path,
    per_class_csv: Path,
    per_file_csv: Path,
    sampled_csv: Path,
    precision_threshold: float,
) -> pd.DataFrame:
    included = set(load_included_categories(included_txt))
    valid_classes = load_valid_classes(per_class_csv, precision_threshold)
    eligible_cats = included & valid_classes
    valid_pairs = load_valid_pairs(per_file_csv, precision_threshold)

    sampled = pd.read_csv(sampled_csv)
    sampled = sampled[sampled["trial_type"] == "regular"].copy()
    sampled["category"] = sampled["category"].astype(str).str.strip().str.lower()
    sampled["stem"] = sampled["stem"].astype(str).str.strip().str.lower()

    mask = sampled.apply(lambda r: (r["category"], r["stem"]) in valid_pairs, axis=1)
    sampled = sampled.loc[mask].copy()
    sampled = sampled[sampled["category"].isin(eligible_cats)].copy()
    return sampled[["category", "path", "stem"]].reset_index(drop=True)


def zscore_rows(X: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    mu = X.mean(axis=0)
    sig = X.std(axis=0)
    return (X - mu) / (sig + eps)


In [3]:
def load_normalized_npy_vector(path: Path) -> np.ndarray | None:
    """Load a single `.npy` embedding and L2-normalize (matches prior image-embedding step)."""
    p = remap_absolute_path(path)
    if not p.is_file():
        return None
    v = np.load(p, mmap_mode="r")
    v = np.asarray(v, dtype=np.float64).ravel()
    n = np.linalg.norm(v)
    if n > 0:
        v = v / n
    return v


def average_precomputed_embeddings_for_categories(
    paths_by_cat: dict[str, list[Path]],
    model_label: str = "embed",
) -> tuple[list[str], np.ndarray, dict]:
    categories = sorted(paths_by_cat.keys())
    diag: dict = {"missing_paths": [], "n_paths": {}, "skipped_categories": []}
    cat_vectors: list[np.ndarray] = []
    categories_out: list[str] = []

    for cat in tqdm(categories, desc=f"Average {model_label} per category"):
        paths_in_order = list(dict.fromkeys(paths_by_cat[cat]))
        vecs: list[np.ndarray] = []
        miss: list[str] = []
        path_iter = tqdm(
            paths_in_order,
            desc=f"{cat} ({len(paths_in_order):,} crops)",
            leave=False,
            disable=len(paths_in_order) < 200,
        )
        for p in path_iter:
            v = load_normalized_npy_vector(p)
            if v is None:
                miss.append(str(remap_absolute_path(p)))
            else:
                vecs.append(v)
        diag["missing_paths"].extend(miss[:20])
        diag["n_paths"][cat] = {
            "total": len(paths_in_order),
            "unique_paths": len(paths_in_order),
            "found": len(vecs),
        }
        if not vecs:
            diag["skipped_categories"].append(cat)
            continue
        mean_vec = np.mean(np.stack(vecs, axis=0), axis=0)
        cat_vectors.append(mean_vec)
        categories_out.append(cat)

    X = np.stack(cat_vectors, axis=0) if cat_vectors else np.zeros((0, 0))
    return categories_out, X, diag


In [4]:
for category_set in CATEGORY_SETS:
    inc_path = CATEGORY_FILES[category_set]
    n_clip_csv_cats = None
    clip_filter_list_stats = None
    babydinov3_paths_by_cat = None
    babydinov3_scan_stats = None
    if category_set == "valid129":
        if not CLIP_FILTER_LIST_PATH.is_file():
            raise FileNotFoundError(
                f"Missing CLIP filter list (embeddings that passed CLIP): {CLIP_FILTER_LIST_PATH} "
                f"(set BV_CLIP_FILTER_LIST or BV_CLIP_FILTER_LIST_THRESHOLD)"
            )
        allowed = set(load_included_categories(inc_path))
        inc_all = sorted(allowed)
        # clip_paths_by_cat, dino_paths_by_cat, scan_stats = collect_paired_npy_paths_from_clip_filter_list(
        #     CLIP_FILTER_LIST_PATH, CLIP_EMBEDDINGS_DIR, DINOV3_EMBEDDINGS_DIR, allowed,
        # )
        if not BABYDINOV3_EMBEDDINGS_DIR.is_dir():
            raise FileNotFoundError(
                f"BabyDINOv3 embeddings dir not found: {BABYDINOV3_EMBEDDINGS_DIR} "
                f"(set BV_BABYDINOV3_EMBEDDINGS_DIR or BV_BABYDINOV3_CHECKPOINT_STEP)"
            )
        babydinov3_paths_by_cat, babydinov3_scan_stats = collect_npy_paths_from_clip_filter_list(
            CLIP_FILTER_LIST_PATH,
            BABYDINOV3_EMBEDDINGS_DIR,
            allowed,
            model_label="babydinov3",
        )
        babydinov3_paths_by_cat = {c: babydinov3_paths_by_cat.get(c, []) for c in inc_all}
        n_paths_total = babydinov3_scan_stats["n_paths_on_disk"]
        n_cat = len(inc_all)
        n_clip_csv_cats = sum(1 for c in inc_all if babydinov3_paths_by_cat[c])
        clip_filter_list_stats = babydinov3_scan_stats
        exemplar_source = "flat_babydinov3_npy_clip_filter_list"
        print(f"\n=== {category_set} (BabyDINOv3 only) ===")
        print("included file:", inc_path)
        print("babydinov3 on disk:", n_paths_total, "categories:", n_cat, "with >=1 npy:", n_clip_csv_cats)
        print("CLIP filter list:", CLIP_FILTER_LIST_PATH)
        print("babydinov3_scan_stats:", babydinov3_scan_stats)
    elif category_set == "valid85":
        exemplar_df = build_valid85_sampled_exemplar_table(
            inc_path,
            PER_CLASS_PRECISION_CSV,
            PER_FILE_PRECISION_CSV,
            SAMPLED_EXEMPLAR_CSV,
            PRECISION_THRESHOLD,
        )
        if not CLIP_FILTER_LIST_PATH.is_file():
            raise FileNotFoundError(f"Missing CLIP filter list: {CLIP_FILTER_LIST_PATH}")
        elig_cats = set(exemplar_df["category"].astype(str).str.strip().str.lower())
        clip_filter_pairs = load_clip_filter_pair_set(CLIP_FILTER_LIST_PATH, elig_cats)
        # clip_paths_by_cat = valid85_npy_paths_by_category_from_manifest(
        #     exemplar_df, CLIP_EMBEDDINGS_DIR, clip_filter_pairs
        # )
        # dino_paths_by_cat = valid85_npy_paths_by_category_from_manifest(
        #     exemplar_df, DINOV3_EMBEDDINGS_DIR, clip_filter_pairs
        # )
        if not BABYDINOV3_EMBEDDINGS_DIR.is_dir():
            raise FileNotFoundError(
                f"BabyDINOv3 embeddings dir not found: {BABYDINOV3_EMBEDDINGS_DIR}"
            )
        babydinov3_paths_by_cat = valid85_npy_paths_by_category_from_manifest(
            exemplar_df, BABYDINOV3_EMBEDDINGS_DIR, clip_filter_pairs
        )
        n_paths_total = sum(len(v) for v in babydinov3_paths_by_cat.values())
        n_cat = len(babydinov3_paths_by_cat)
        n_clip_csv_cats = sum(1 for v in babydinov3_paths_by_cat.values() if v)
        clip_filter_list_stats = {
            "n_category_stem_pairs_in_clip_filter_list_for_eligible_cats": len(clip_filter_pairs),
        }
        exemplar_source = "sampled_csv_validated_clip_filter_list_babydinov3_npy"
        print(f"\n=== {category_set} (BabyDINOv3 only) ===")
        print("included file:", inc_path)
        print("CLIP filter list:", CLIP_FILTER_LIST_PATH)
        print(
            "category/stem pairs from CLIP filter list covering eligible manifest categories:",
            len(clip_filter_pairs),
        )
        print("sampled manifest rows:", len(exemplar_df), "unique categories:", exemplar_df["category"].nunique())
        print("unique .npy paths after dedupe:", n_paths_total, "categories in paths map:", n_cat)
    else:
        raise ValueError(category_set)

    out_dir = OUT_ROOT / category_set
    out_dir.mkdir(parents=True, exist_ok=True)

    # cats_c, Xc_raw, diag_c = average_precomputed_embeddings_for_categories(
    #     clip_paths_by_cat, model_label="CLIP"
    # )
    # cats_d, Xd_raw, diag_d = average_precomputed_embeddings_for_categories(
    #     dino_paths_by_cat, model_label="DINOv3"
    # )
    # save_cat_emb(... bv_clip_*, bv_dinov3_* ...)

    def save_cat_emb(csv_path: Path, cats: list[str], X: np.ndarray, dim_cols: list[str]) -> None:
        df = pd.DataFrame(X, columns=dim_cols)
        df.insert(0, "category", cats)
        df.to_csv(csv_path, index=False)

    if babydinov3_paths_by_cat is None:
        raise RuntimeError("babydinov3_paths_by_cat was not built")
    cats_bd, Xbd_raw, diag_bd = average_precomputed_embeddings_for_categories(
        babydinov3_paths_by_cat, model_label="BabyDINOv3"
    )
    Xbd_z = zscore_rows(Xbd_raw)
    dim_cols_bd = [f"dim_{i}" for i in range(Xbd_z.shape[1])]
    save_cat_emb(
        out_dir / f"bv_babydinov3_exemplar_avg_zscore_within_{category_set}.csv",
        cats_bd,
        Xbd_z,
        dim_cols_bd,
    )
    save_cat_emb(
        out_dir / f"bv_babydinov3_exemplar_avg_raw_within_{category_set}.csv",
        cats_bd,
        Xbd_raw,
        dim_cols_bd,
    )

    meta = {
        "category_set": category_set,
        "run_mode": "babydinov3_only",
        "exemplar_source": exemplar_source,
        "included_categories_txt": str(inc_path),
        "n_categories": len(cats_bd),
        "n_unique_npy_paths": n_paths_total,
        "n_inclusion_categories": n_cat,
        "n_categories_with_at_least_one_npy": n_clip_csv_cats,
        "precision_threshold": PRECISION_THRESHOLD,
        "clip_filter_list_path": str(CLIP_FILTER_LIST_PATH),
        "clip_filter_list_stats": clip_filter_list_stats,
        "babydinov3_checkpoint_step": BABYDINOV3_CHECKPOINT_STEP,
        "babydinov3_embeddings_dir": str(BABYDINOV3_EMBEDDINGS_DIR),
        "categories_alphabetical": cats_bd,
        "diagnostics_babydinov3": {k: v for k, v in diag_bd.items() if k != "missing_paths"},
        "n_missing_paths_reported_babydinov3": len(diag_bd.get("missing_paths", [])),
    }
    (out_dir / "exemplar_embedding_run.json").write_text(json.dumps(meta, indent=2))
    print("Wrote:", out_dir / f"bv_babydinov3_exemplar_avg_zscore_within_{category_set}.csv")
    print("Wrote:", out_dir / f"bv_babydinov3_exemplar_avg_raw_within_{category_set}.csv")
    print("Wrote:", out_dir / "exemplar_embedding_run.json")

print("\nDone.")


Index babydinov3 category dirs: 100%|██████████| 129/129 [00:00<00:00, 72626.20it/s]
Scan CLIP filter list (babydinov3): 4183889 lines [02:44, 25490.68 lines/s]


  [CLIP filter list → babydinov3] lines=4,183,889 in_allowed_cats=3,625,482 on_disk=3,604,656 missing_file_or_dir=20,826

=== valid129 (BabyDINOv3 only) ===
included file: /home/j7yang/babyview-projects/vss2026/object-detection/data/included_categories_valid129.txt
babydinov3 on disk: 3604656 categories: 129 with >=1 npy: 129
CLIP filter list: /data2/dataset/babyview/868_hours/outputs/yoloe_cdi_embeddings/clip_image_embeddings_filtered-by-clip-0.27_exclude-people_exclude-subject-00270001.txt
babydinov3_scan_stats: {'model_label': 'babydinov3', 'n_lines_in_clip_filter_list': 4183889, 'n_lines_in_allowed_categories': 3625482, 'n_paths_on_disk': 3604656, 'n_lines_missing_file_or_dir': 20826, 'categories_with_subdir': 129}


Average BabyDINOv3 per category: 100%|██████████| 129/129 [49:05<00:00, 22.83s/it]  


Wrote: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129/bv_babydinov3_exemplar_avg_zscore_within_valid129.csv
Wrote: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129/bv_babydinov3_exemplar_avg_raw_within_valid129.csv
Wrote: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/manuscript-2026/exemplar_set_embeddings/valid129/exemplar_embedding_run.json

Done.
